# EMG Signal Preprocessing Pipeline

Based on `1.0_analyze.ipynb` and `1.0_signal-verification.ipynb`

**Preprocessing steps:**
1. No baseline correction (no subtraction)
2. Bandpass filter (20-500 Hz)
3. Notch filter at 50 Hz
4. Segment into 6 intervals (baseline, close_0→open_0, open_0→close_1, ...)
5. Keep only **interval 0 (baseline)** — first 5 seconds, no movement — for PSD computation
6. Exclude Sample 1 (exp_1) within each block — not a recovery condition
7. Compute Power Spectral Density (PSD) on baseline segment
8. Extract frequency-domain features (1-450 Hz for each channel)

**Output:** `dataset_filtered_1_450.csv` compatible with `dt_classification_grid_search.ipynb`

In [1]:
import json
import os
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt, iirnotch, welch

In [2]:
# Configuration
DATA_DIR = "/Users/shikaiyi/.cache/kagglehub/datasets/levshaazz/emg-muscle-recovery/versions/1/data"
OUTPUT_PATH = "dataset_filtered_1_450.csv"
SAMPLE_RATE = 1000  # Hz
BANDPASS_LOW = 20  # Hz
BANDPASS_HIGH = 500  # Hz
NOTCH_FREQ = 50  # Hz
FREQ_RANGE = (1, 450)  # Hz for PSD features
N_CHANNELS = 3
REST_TIME_MAP = {"1min": 0, "5min": 1, "10min": 2}

In [3]:
def load_and_preprocess(
    filepath: str,
    sfreq: int = SAMPLE_RATE,
    bandpass_low: int = BANDPASS_LOW,
    bandpass_high: int = BANDPASS_HIGH,
    notch_freq: int = NOTCH_FREQ,
) -> Tuple[np.ndarray, np.ndarray, Dict]:
    """
    Load EMG data, apply bandpass and notch filters.
    Uses first 5 seconds of each sample as baseline period (no movement) — no baseline correction.
    Returns: (time, filtered_data, points)
    """
    df = pd.read_csv(f"{filepath}.csv")
    time = df.iloc[:, 0].values  # First column: time
    data = df.iloc[:, 1:4].values.astype(float)  # emg1, emg2, emg3
    
    # Convert from raw ADC if needed (values 0-65535)
    if np.max(data) > 10:
        data = data * 3.3 / 65535
    
    with open(f"{filepath}_points.json", "r") as f:
        points = json.load(f)
    
    # 1. No baseline correction. Keep full signal including baseline segment [0, baseline]
    end_idx = np.searchsorted(time, 22.0)
    if end_idx >= len(time):
        end_idx = len(time)
    time = time[:end_idx]
    data = data[:end_idx, :]
    
    # Resample to uniform sampling if needed (for Welch method)
    duration = time[-1] - time[0]
    n_samples = int(duration * sfreq)
    if abs(len(time) - n_samples) > n_samples * 0.1:  # >10% difference
        from scipy.interpolate import interp1d
        time_uniform = np.linspace(time[0], time[-1], n_samples)
        data_resampled = np.zeros((n_samples, N_CHANNELS))
        for ch in range(N_CHANNELS):
            f_interp = interp1d(time, data[:, ch], kind='linear', fill_value='extrapolate')
            data_resampled[:, ch] = f_interp(time_uniform)
        time, data = time_uniform, data_resampled
    
    # 2. Bandpass filter (20-500 Hz)
    nyq = sfreq / 2
    low = bandpass_low / nyq
    high = min(bandpass_high / nyq, 0.99)
    b, a = butter(4, [low, high], btype="bandpass")
    for ch in range(N_CHANNELS):
        data[:, ch] = filtfilt(b, a, data[:, ch])
    
    # 3. Notch filter at 50 Hz
    b_notch, a_notch = iirnotch(notch_freq, 30.0, sfreq)
    for ch in range(N_CHANNELS):
        data[:, ch] = filtfilt(b_notch, a_notch, data[:, ch])
    
    return time, data, points

In [4]:
def segment_signal(
    time: np.ndarray,
    data: np.ndarray,
    points: Dict,
) -> List[Tuple[np.ndarray, np.ndarray, str]]:
    """
    Segment signal into 6 intervals according to event markers.
    Returns list of (time_seg, data_seg, label) for each segment.
    Interval 0: [0, baseline] — baseline (first 5s, no movement). Use this for PSD.
    Intervals 1-5: close_0->open_0, open_0->close_1, close_1->open_1, open_1->close_2, close_2->open_2.
    """
    segments = []
    baseline_end = points["baseline"]

    # Interval 0: [0, baseline] — baseline (first 5s, no movement)
    idx_end = np.searchsorted(time, baseline_end)
    idx_end = min(idx_end, len(time))
    if idx_end > 0:
        segments.append((time[:idx_end], data[:idx_end, :], "baseline"))

    # Intervals 1-5: between consecutive markers
    bounds = [
        ("close_0", "open_0"),
        ("open_0", "close_1"),
        ("close_1", "open_1"),
        ("open_1", "close_2"),
        ("close_2", "open_2"),
    ]
    for t_start_key, t_end_key in bounds:
        t_start = points[t_start_key]
        t_end = points[t_end_key]
        idx_start = np.searchsorted(time, t_start)
        idx_end = np.searchsorted(time, t_end)
        if idx_start < idx_end:
            segments.append((time[idx_start:idx_end], data[idx_start:idx_end, :], f"{t_start_key}-{t_end_key}"))
    return segments

In [5]:
def compute_psd_features_v2(
    data_seg: np.ndarray,
    sfreq: int = SAMPLE_RATE,
    freq_range: Tuple[int, int] = FREQ_RANGE,
) -> Dict[str, float]:
    """
    Compute PSD using Welch and extract 1-450 Hz features per channel.
    Maps Welch output frequencies to integer Hz bins.
    """
    nperseg = min(512, len(data_seg) // 2)
    if nperseg < 8:
        return None
    
    result = {}
    for ch in range(N_CHANNELS):
        f, psd = welch(data_seg[:, ch], fs=sfreq, nperseg=nperseg)
        for freq_hz in range(freq_range[0], freq_range[1] + 1):
            idx = np.argmin(np.abs(f - freq_hz))
            result[f"ch_{ch+1}_{freq_hz}"] = float(psd[idx])
    return result

In [6]:
def process_all_data(data_dir: str = DATA_DIR) -> pd.DataFrame:
    """Process all EMG files and build dataset."""
    rows = []
    skipped = []
    
    for user_id in sorted(os.listdir(data_dir)):
        user_path = Path(data_dir) / user_id
        if not user_path.is_dir() or not user_id.isdigit():
            continue
        
        for rest_time in ["1min", "5min", "10min"]:
            rest_path = user_path / rest_time
            if not rest_path.exists():
                continue
            
            rest_int = REST_TIME_MAP[rest_time]
            
            for exp_num in [1, 2, 3]:
                # Exclude Sample 1 (exp_1) — not a recovery condition
                if exp_num == 1:
                    continue
                csv_file = rest_path / f"exp_{exp_num}.csv"
                json_file = rest_path / f"exp_{exp_num}_points.json"
                if "broken" in str(csv_file):
                    continue
                if not csv_file.exists() or not json_file.exists():
                    skipped.append(str(csv_file))
                    continue
                filepath = str(rest_path / f"exp_{exp_num}")
                
                try:
                    time, data, points = load_and_preprocess(str(filepath))
                    segments = segment_signal(time, data, points)
                    
                    # Keep only seq 0 (baseline), exclude exp_1
                    for seq, (seg_time, seg_data, label) in enumerate(segments):
                        if seq != 0:
                            continue
                        features = compute_psd_features_v2(seg_data)
                        if features is None:
                            continue
                        row = {
                            "user_id": int(user_id),
                            "rest_int": rest_int,
                            "exp": exp_num,
                            "seq": seq,
                            **features
                        }
                        rows.append(row)
                except Exception as e:
                    skipped.append(f"{filepath}: {e}")
    
    if skipped:
        print(f"Skipped {len(skipped)} files:", skipped[:5], "...")
    
    return pd.DataFrame(rows)

In [7]:
# Run preprocessing pipeline
print("Processing EMG data...")
df = process_all_data()
print(f"Processed {len(df)} samples")
print(f"Columns: {list(df.columns[:10])}... ({len(df.columns)} total)")
df.head()

Processing EMG data...
Skipped 3 files: ['/Users/shikaiyi/.cache/kagglehub/datasets/levshaazz/emg-muscle-recovery/versions/1/data/3/1min/exp_3.csv', '/Users/shikaiyi/.cache/kagglehub/datasets/levshaazz/emg-muscle-recovery/versions/1/data/5/1min/exp_2.csv', '/Users/shikaiyi/.cache/kagglehub/datasets/levshaazz/emg-muscle-recovery/versions/1/data/6/5min/exp_3.csv'] ...
Processed 81 samples
Columns: ['user_id', 'rest_int', 'exp', 'seq', 'ch_1_1', 'ch_1_2', 'ch_1_3', 'ch_1_4', 'ch_1_5', 'ch_1_6']... (1354 total)


,user_id,rest_int,exp,seq,ch_1_1,ch_1_2,ch_1_3,ch_1_4,ch_1_5,ch_1_6,...,ch_3_441,ch_3_442,ch_3_443,ch_3_444,ch_3_445,ch_3_446,ch_3_447,ch_3_448,ch_3_449,ch_3_450
0,1,0,2,0,1.005382e-06,1.005382e-06,3.472351e-10,3.472351e-10,9.595464e-10,9.595464e-10,...,9.960482e-07,9.960482e-07,1.728703e-06,1.728703e-06,1.874990e-06,1.874990e-06,2.501966e-06,2.501966e-06,2.440420e-06,2.440420e-06
1,1,0,3,0,4.590699e-06,4.590699e-06,1.524460e-09,1.524460e-09,5.619545e-09,5.619545e-09,...,7.198383e-07,7.198383e-07,7.309644e-07,7.309644e-07,6.288736e-07,6.288736e-07,8.313117e-07,8.313117e-07,7.164160e-07,7.164160e-07
2,1,1,2,0,6.877400e-07,6.877400e-07,3.641048e-10,3.641048e-10,1.115864e-09,1.115864e-09,...,2.318988e-06,2.318988e-06,2.343714e-06,2.343714e-06,2.916858e-06,2.916858e-06,1.908059e-06,1.908059e-06,1.893212e-06,1.893212e-06
3,1,1,3,0,9.469974e-07,9.469974e-07,2.426888e-10,2.426888e-10,6.061452e-10,6.061452e-10,...,3.174626e-06,3.174626e-06,2.685582e-06,2.685582e-06,2.224423e-06,2.224423e-06,2.196489e-06,2.196489e-06,2.547083e-06,2.547083e-06
4,1,2,2,0,1.133963e-06,1.133963e-06,4.005027e-10,4.005027e-10,1.100986e-09,1.100986e-09,...,1.085708e-06,1.085708e-06,9.645583e-07,9.645583e-07,9.490736e-07,9.490736e-07,1.282742e-06,1.282742e-06,9.827336e-07,9.827336e-07


In [8]:
# Save preprocessed dataset
df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved to {OUTPUT_PATH}")
print(f"Shape: {df.shape}")
print(f"Feature columns (excluding user_id, rest_int, exp, seq): {len(df.columns) - 4}")

Saved to dataset_filtered_1_450.csv
Shape: (81, 1354)
Feature columns (excluding user_id, rest_int, exp, seq): 1350


In [9]:
# Verify compatibility with dt_classification_grid_search
REMOVE_LIST = ["user_id", "rest_int", "exp", "seq"]
features_list = [c for c in df.columns if c not in REMOVE_LIST]
print(f"Number of features: {len(features_list)}")
print(f"Expected: 1350 (3 channels x 450 frequencies)")
print(f"rest_int unique values: {sorted(df['rest_int'].unique())}")

Number of features: 1350
Expected: 1350 (3 channels x 450 frequencies)
rest_int unique values: [np.int64(0), np.int64(1), np.int64(2)]


In [10]:
# Quick verification: load output and check compatibility with dt_classification_grid_search
try:
    check_df = pd.read_csv(OUTPUT_PATH)
    remove_list = ["user_id", "rest_int", "exp", "seq"]
    features_list = [c for c in check_df.columns if c not in remove_list]
    print("Verification passed!")
    print(f"  - Rows: {len(check_df)}")
    print(f"  - Features: {len(features_list)} (expected 1350)")
    print(f"  - rest_int values: {sorted(check_df['rest_int'].unique())}")
    print(f"  - Sample feature names: {features_list[:5]} ... {features_list[-3:]}")
except FileNotFoundError:
    print("Run the pipeline above first to generate", OUTPUT_PATH)

Verification passed!
  - Rows: 81
  - Features: 1350 (expected 1350)
  - rest_int values: [np.int64(0), np.int64(1), np.int64(2)]
  - Sample feature names: ['ch_1_1', 'ch_1_2', 'ch_1_3', 'ch_1_4', 'ch_1_5'] ... ['ch_3_448', 'ch_3_449', 'ch_3_450']


In [11]:
# Dimensionality Reduction: Log-Band Grouping
# Frequency bands: 0-4, 4-8, 8-16, 16-32, 32-64, 64-128, 128-256 Hz
# Output: 3 channels × 7 bands = 21 features per segment

OUTPUT_BAND_PATH = "dataset_filtered_bands.csv"
EPS = 1e-12  # avoid log(0)

# Band definitions: (low_freq, high_freq) inclusive, maps to ch_X_Y (1-indexed Hz)
BANDS = [
    (1, 4),     # 0-4 Hz
    (5, 8),     # 4-8 Hz
    (9, 16),    # 8-16 Hz
    (17, 32),   # 16-32 Hz
    (33, 64),   # 32-64 Hz
    (65, 128),  # 64-128 Hz
    (129, 256), # 128-256 Hz
]

# Load full PSD dataset
df_full = pd.read_csv(OUTPUT_PATH)
meta_cols = ["user_id", "rest_int", "exp", "seq"]
df_meta = df_full[meta_cols].copy()

# Aggregate per band: mean PSD within band, then log
band_cols = []
for ch in range(1, 4):
    for b_idx, (low, high) in enumerate(BANDS):
        cols = [f"ch_{ch}_{f}" for f in range(low, high + 1)]
        band_mean = df_full[cols].mean(axis=1)
        s = np.log(band_mean + EPS)
        s.name = f"ch_{ch}_band_{b_idx + 1}"
        band_cols.append(s)

band_df = pd.concat([df_meta] + band_cols, axis=1)
band_df.to_csv(OUTPUT_BAND_PATH, index=False)
print(f"Saved to {OUTPUT_BAND_PATH}")
print(f"Shape: {band_df.shape}")
print(f"Features: 3 channels × 7 bands = 21 (expected)")
print(f"Sample columns: {list(band_df.columns[:8])}...")
band_df.head()

Saved to dataset_filtered_bands.csv
Shape: (81, 25)
Features: 3 channels × 7 bands = 21 (expected)
Sample columns: ['user_id', 'rest_int', 'exp', 'seq', 'ch_1_band_1', 'ch_1_band_2', 'ch_1_band_3', 'ch_1_band_4']...


,user_id,rest_int,exp,seq,ch_1_band_1,ch_1_band_2,ch_1_band_3,ch_1_band_4,ch_1_band_5,ch_1_band_6,...,ch_2_band_5,ch_2_band_6,ch_2_band_7,ch_3_band_1,ch_3_band_2,ch_3_band_3,ch_3_band_4,ch_3_band_5,ch_3_band_6,ch_3_band_7
0,1,0,2,0,-14.502942,-18.608048,-10.606471,-6.784710,-7.456540,-8.948797,...,-7.114605,-8.765760,-10.769308,-14.551530,-18.500930,-10.955579,-6.738123,-7.651262,-9.117838,-11.075607
1,1,0,3,0,-12.984293,-17.151432,-9.582711,-5.918418,-6.537837,-8.260686,...,-8.548311,-10.260122,-12.324193,-15.427467,-19.630675,-11.759565,-7.864883,-8.730231,-10.302195,-12.341754
2,1,1,2,0,-14.882470,-18.959739,-10.772566,-7.334048,-7.950068,-9.473869,...,-8.337007,-9.873608,-11.802961,-14.171820,-18.459678,-10.474835,-6.689189,-7.387068,-9.097615,-10.921538
3,1,1,3,0,-14.562858,-18.995733,-11.038280,-7.292388,-7.883258,-9.731391,...,-8.307322,-9.843671,-11.592113,-14.067133,-18.339164,-10.961358,-6.878911,-7.601748,-9.012193,-10.850714
4,1,2,2,0,-14.382584,-18.537001,-10.909938,-7.063371,-7.951636,-9.633207,...,-6.900264,-8.375263,-10.893462,-15.345260,-18.562046,-10.984869,-7.100589,-7.973253,-9.495180,-11.909357
